# Notebook v2 — menyatukan semua eksperimen CIC-IDS2017 pada data yang sama + uji mekanisme

**Mengapa v2?** Hasil v1 memakai rilis resmi `MachineLearningCSV` (total 124.127 record setelah dibatasi), sedangkan Tabel 1, 2, 6 dan Fig. 2 di naskah berasal dari versi data lama (95.572 record). Satu paper tidak boleh mencampur dua versi data, jadi semua tabel CIC-IDS2017 dijalankan ulang di sini dengan pipeline yang sama.

**Tambahan penting untuk novelty:**
* **Random oversampling (ROS)** pada kondisi n = 29. ROS hanya menduplikasi sampel tanpa membuat titik baru. Jika manfaat ROS ≈ SMOTE, berarti keuntungan SMOTE berasal dari pergeseran prior/batas keputusan, bukan dari geometri titik sintetis.
* **DoS dan DDoS** ikut dipangkas ke 29 sampel, sehingga ada 6 kategori (sebelumnya 4).
* **Bentuk distribusi dihitung untuk setiap subsampel 29** yang benar-benar dipakai melatih model, sehingga bisa dikorelasikan langsung dengan manfaat SMOTE.
* Kategori tujuan salah prediksi (false negative) dicatat.

**Checkpoint v1 dipakai ulang.** Salin tidak perlu: notebook ini membaca folder `Drive/juti_rerun` yang sama, jadi model yang sudah selesai di v1 tidak dilatih ulang.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| 0–3 | Setup, unduh, bersihkan, split | ±15 menit |
| A | Tabel 1 + Tabel 6 (8 algoritma, 80:20) | ±40–60 menit |
| B | Bentuk distribusi 6 kategori | ±5 menit |
| C | Eksperimen terkontrol v2 (6 kategori × base/SMOTE/ROS) | ±3 jam (melanjutkan v1) |
| D | Tabel 4 lengkap (semua metode + Isolation Forest) | ±2 jam (melanjutkan v1) |
| E | CSE-CIC-IDS2018: kurva belajar Infiltration (Tabel 5) | ±40 menit |
| F | Ringkasan tabel + gambar siap naskah | < 2 menit |

Semua bagian bisa dijalankan terpisah lewat saklar `RUN_...` di sel pengaturan. Jika Colab terputus, jalankan ulang dari atas; bagian yang sudah selesai akan dilewati.

## 0. Pengaturan

In [ ]:
# ====== PENGATURAN (boleh diubah) ======
SEEDS           = [42, 43, 44]  # jumlah seed untuk noise floor; mulai dengan [42] jika ingin cepat
N_FOLDS         = 5             # stratified k-fold
CAP             = 30000         # batas maksimum record per kategori (sesuai naskah)
TARGET          = 24000         # target oversampling per kelas (sesuai naskah)
N_CONTROL       = 29            # titik kontrol (jumlah sampel latih Infiltration)
RF_TREES        = 100           # jumlah pohon Random Forest
RUN_ALL_METHODS = True          # semua konfigurasi Tabel 4
RANDOM_STATE    = 42
TEST_MODE       = False         # biarkan False (hanya untuk uji internal)

RUN_A_TABLE1    = True          # Tabel 1 + Tabel 6
RUN_B_SHAPE     = True          # bentuk distribusi 6 kategori
RUN_C_CONTROLLED= True          # eksperimen terkontrol v2
RUN_D_METHODS   = True          # Tabel 4 lengkap
RUN_E_CSE2018   = True          # kurva belajar CSE-CIC-IDS2018
TARGET_CATS     = ["Botnet", "PortScan", "Web Attack", "Infiltration", "DoS", "DDoS"]

import os, time, json, math, zipfile, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/juti_rerun"
except Exception:
    OUTDIR = "./juti_rerun"
os.makedirs(OUTDIR, exist_ok=True)
DATADIR = "/content/data" if os.path.isdir("/content") else "./data"
os.makedirs(DATADIR, exist_ok=True)
print("Hasil akan disimpan di:", OUTDIR)

Mounted at /content/drive
Hasil akan disimpan di: /content/drive/MyDrive/juti_rerun


In [ ]:
# Pasang library dengan versi yang tercatat di naskah (Sec. 3.6)
import subprocess, sys
if not TEST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.6.1", "imbalanced-learn==0.14.2", "xgboost", "boto3"], check=False)
import sklearn, imblearn
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__)
print("Jika versi di atas belum 1.6.1 / 0.14.2: menu Runtime > Restart session, lalu jalankan ulang dari atas.")
import xgboost; print("xgboost", xgboost.__version__)

scikit-learn 1.6.1 | imbalanced-learn 0.14.2
Jika versi di atas belum 1.6.1 / 0.14.2: menu Runtime > Restart session, lalu jalankan ulang dari atas.
xgboost 3.4.1


## 1. Unduh CIC-IDS2017

In [ ]:
ZIP_PATH = os.path.join(DATADIR, "MachineLearningCSV.zip")
SOURCES = [
    "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCSV.zip",
    "https://huggingface.co/datasets/Mireu-Lab/CIC-IDS/resolve/main/ZIP/MachineLearningCSV.zip",
]
drive_copy = os.path.join(OUTDIR, "MachineLearningCSV.zip")

def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return any(n.endswith(".csv") for n in z.namelist())
    except Exception:
        return False

if not TEST_MODE:
    if not zip_ok(ZIP_PATH) and zip_ok(drive_copy):
        import shutil; shutil.copy(drive_copy, ZIP_PATH)
    for url in SOURCES:
        if zip_ok(ZIP_PATH):
            break
        print("Mengunduh dari", url)
        subprocess.run(["wget", "-q", "--tries=2", "--timeout=60", "-O", ZIP_PATH, url])
    if not zip_ok(ZIP_PATH):
        raise RuntimeError("Unduhan gagal. Unggah MachineLearningCSV.zip ke Drive/juti_rerun lalu jalankan ulang sel ini.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATADIR)
CSV_FILES = sorted(glob.glob(os.path.join(DATADIR, "**", "*.csv"), recursive=True))
print(len(CSV_FILES), "file CSV ditemukan")
for f in CSV_FILES: print(" -", os.path.basename(f))

Mengunduh dari http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCSV.zip
Mengunduh dari https://huggingface.co/datasets/Mireu-Lab/CIC-IDS/resolve/main/ZIP/MachineLearningCSV.zip
8 file CSV ditemukan
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


## 2. Pembersihan data

In [ ]:
def category_of(label):
    l = str(label).strip()
    if l == "BENIGN": return "BENIGN"
    if l == "DDoS": return "DDoS"
    if l.startswith("DoS"): return "DoS"
    if l == "PortScan": return "PortScan"
    if l.startswith("Web Attack"): return "Web Attack"
    if l == "Bot": return "Botnet"
    if l == "Infiltration": return "Infiltration"
    if l in ("FTP-Patator", "SSH-Patator"): return "Brute Force"
    if l == "Heartbleed": return "Heartbleed"
    return None

frames = []
for f in CSV_FILES:
    d = pd.read_csv(f, encoding="latin-1", low_memory=False)
    d.columns = [c.strip() for c in d.columns]
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)
print("Record mentah:", f"{len(raw):,}")

raw = raw.drop_duplicates()
print("Setelah hapus duplikat:", f"{len(raw):,}")

label = raw["Label"]
X = raw.drop(columns=["Label"])
del raw, frames
obj_cols = X.select_dtypes(include="object").columns
for c_ in obj_cols:                      # kolom non-numerik diubah; yang gagal jadi null
    X[c_] = pd.to_numeric(X[c_], errors="coerce")
X = X.loc[:, X.notna().any()]            # buang kolom yang seluruhnya non-numerik
X = X.replace([np.inf, -np.inf], np.nan)
keep = X.notna().all(axis=1)
X, label = X[keep], label[keep]
print("Setelah hapus baris inf/null:", f"{len(X):,}")

cat = label.map(category_of)
print("\nJumlah per label asli yang dibuang:")
print(cat[cat.isin(["Brute Force", "Heartbleed"])].value_counts().to_string())
mask = cat.isin(["BENIGN", "DoS", "DDoS", "PortScan", "Web Attack", "Botnet", "Infiltration"])
X, cat = X[mask].reset_index(drop=True), cat[mask].reset_index(drop=True)

const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)
col_hash = {c_: pd.util.hash_pandas_object(X[c_], index=False).values.sum() for c_ in X.columns}
seen, dup_cols = {}, []
for c_ in X.columns:                     # kolom yang isinya identik dengan kolom sebelumnya
    h = col_hash[c_]
    if h in seen and X[c_].equals(X[seen[h]]):
        dup_cols.append(c_)
    else:
        seen.setdefault(h, c_)
X = X.drop(columns=dup_cols)
print("\nKolom konstan dihapus  (naskah: 8):", len(const_cols), const_cols)
print("Kolom kembar dihapus :", dup_cols)
print("Jumlah fitur akhir    (naskah: 69):", X.shape[1])

print("\nJumlah per kategori sebelum dibatasi:")
print(cat.value_counts().to_string())

rng_cap = np.random.RandomState(RANDOM_STATE)
idx = []
for c, grp in cat.groupby(cat):
    ids = grp.index.values
    if len(ids) > CAP:
        ids = rng_cap.choice(ids, size=CAP, replace=False)
    idx.extend(ids)
idx = np.sort(np.array(idx))
X, cat = X.loc[idx].reset_index(drop=True), cat.loc[idx].reset_index(drop=True)
print("\nSetelah dibatasi (naskah total: 95.572):", f"{len(X):,}")
print(cat.value_counts().to_string())

FEATURES = X.columns.tolist()
X_all = X.values.astype(np.float64)
y_all = cat.values.astype(str)
CLASSES = sorted(np.unique(y_all))
pd.DataFrame({"n": cat.value_counts()}).to_csv(os.path.join(OUTDIR, "counts_after_cap.csv"))

summary_pre = {"n_raw_after_dedup_and_inf": int(len(X)), "n_features": len(FEATURES),
               "constant_columns": const_cols, "duplicate_columns": dup_cols,
               "counts_after_cap": cat.value_counts().to_dict(), "total_after_cap": int(len(X_all))}
json.dump(summary_pre, open(os.path.join(OUTDIR, "preprocessing_summary.json"), "w"), indent=2)

Record mentah: 2,830,743
Setelah hapus duplikat: 2,522,362
Setelah hapus baris inf/null: 2,520,798

Jumlah per label asli yang dibuang:
Label
Brute Force    9150
Heartbleed       11

Kolom konstan dihapus  (naskah: 8): 8 ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Kolom kembar dihapus : ['SYN Flag Count', 'CWE Flag Count', 'Fwd Header Length.1', 'Subflow Fwd Packets', 'Subflow Bwd Packets']
Jumlah fitur akhir    (naskah: 69): 65

Jumlah per kategori sebelum dibatasi:
Label
BENIGN          2095057
DoS              193745
DDoS             128014
PortScan          90694
Web Attack         2143
Botnet             1948
Infiltration         36

Setelah dibatasi (naskah total: 95.572): 124,127
Label
BENIGN          30000
DDoS            30000
PortScan        30000
DoS             30000
Web Attack       2143
Botnet           1948
Infiltration       36


## 3. Pembagian 80:20 dan standardisasi

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_tr_raw)          # hanya dari data latih
X_tr = scaler.transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)
print("Latih:", X_tr.shape, "| Uji:", X_te.shape)
print("Infiltration latih/uji:", (y_tr == "Infiltration").sum(), "/", (y_te == "Infiltration").sum())

Latih: (99301, 65) | Uji: (24826, 65)
Infiltration latih/uji: 29 / 7


## A. Tabel 1 (8 algoritma) dan Tabel 6 (recall Infiltration pada split 80:20)

Semua algoritma memakai split dan fitur yang sama dari bagian 3. Waktu latih dicatat dalam detik; angkanya bergantung pada mesin Colab, jadi di naskah cukup dilaporkan sebagai perbandingan relatif.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

CKA = os.path.join(OUTDIR, "secA_table1_table6.csv")
doneA = pd.read_csv(CKA) if os.path.exists(CKA) else pd.DataFrame()
ALGOS = {
    "XGBoost": lambda: XGBClassifier(n_estimators=RF_TREES, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": lambda: GradientBoostingClassifier(n_estimators=RF_TREES, random_state=RANDOM_STATE),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=RANDOM_STATE),
    "Decision Tree": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Extra Trees": lambda: ExtraTreesClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Linear SVC": lambda: LinearSVC(max_iter=5000, random_state=RANDOM_STATE),
    "Naive Bayes": lambda: GaussianNB(),
}
if RUN_A_TABLE1:
    enc = {c: i for i, c in enumerate(CLASSES)}
    ytr_i, yte_i = np.array([enc[v] for v in y_tr]), np.array([enc[v] for v in y_te])
    for name, make in ALGOS.items():
        if len(doneA) and name in set(doneA["algorithm"]):
            continue
        t0 = time.time()
        m = make().fit(X_tr, ytr_i)
        t_fit = time.time() - t0
        pred_i = m.predict(X_te)
        pred = np.array(CLASSES)[pred_i]
        inf_te = y_te == "Infiltration"
        row = dict(algorithm=name, accuracy=100 * accuracy_score(yte_i, pred_i),
                   macro_f1=100 * f1_score(yte_i, pred_i, average="macro"), train_time_s=t_fit,
                   inf_test=int(inf_te.sum()), inf_correct=int((pred[inf_te] == "Infiltration").sum()),
                   inf_fp=int(((pred == "Infiltration") & ~inf_te).sum()),
                   inf_fn_to_benign=int((pred[inf_te] == "BENIGN").sum()))
        doneA = pd.concat([doneA, pd.DataFrame([row])], ignore_index=True)
        doneA.to_csv(CKA, index=False)
        print(f"{name:20s} acc {row['accuracy']:.2f}  macroF1 {row['macro_f1']:.1f}  {t_fit:.0f} s  "
              f"Infiltration {row['inf_correct']}/{row['inf_test']}")

XGBoost              acc 99.90  macroF1 98.6  35 s  Infiltration 6/7
Gradient Boosting    acc 99.81  macroF1 97.6  721 s  Infiltration 6/7
Random Forest        acc 99.81  macroF1 97.2  20 s  Infiltration 5/7
Decision Tree        acc 99.76  macroF1 95.8  3 s  Infiltration 6/7
Extra Trees          acc 99.76  macroF1 96.9  7 s  Infiltration 5/7
K-Nearest Neighbors  acc 99.22  macroF1 92.6  0 s  Infiltration 5/7
Linear SVC           acc 96.64  macroF1 79.2  555 s  Infiltration 2/7
Naive Bayes          acc 90.34  macroF1 71.8  0 s  Infiltration 6/7


## B. Bentuk distribusi 6 kategori pada n = 29

Sama seperti v1, ditambah DoS dan DDoS. Fungsi di sel ini juga dipakai bagian C.

In [ ]:
from sklearn.neighbors import NearestNeighbors

K_NN, REPEATS_SHAPE, N_SYNTH = 5, 20, 2000
rng_shape = np.random.default_rng(RANDOM_STATE)

def example_types(X_min, X_other, k=K_NN):
    X_both = np.vstack([X_min, X_other])
    is_min = np.r_[np.ones(len(X_min), bool), np.zeros(len(X_other), bool)]
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_both).kneighbors(X_min)
    same = is_min[ind[:, 1:]].sum(axis=1)
    return {"safe (%)": np.mean(same >= 4) * 100,
            "borderline (%)": np.mean((same == 2) | (same == 3)) * 100,
            "rare (%)": np.mean(same == 1) * 100,
            "outlier (%)": np.mean(same == 0) * 100}

def dispersion_ratio(X_min, nn_other):
    d_in, _ = NearestNeighbors(n_neighbors=2).fit(X_min).kneighbors(X_min)
    d_out, _ = nn_other.kneighbors(X_min, n_neighbors=1)
    return d_in[:, 1].mean() / max(d_out[:, 0].mean(), 1e-12)

def synthetic_intrusion(X_min, nn_other, rng, k=K_NN, n_new=N_SYNTH):
    k = min(k, len(X_min) - 1)
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)
    j = ind[i, rng.integers(1, k + 1, n_new)]
    lam = rng.random((n_new, 1))
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])          # persamaan (1)
    d_min, _ = NearestNeighbors(n_neighbors=1).fit(X_min).kneighbors(X_new)
    d_oth, _ = nn_other.kneighbors(X_new, n_neighbors=1)
    return np.mean(d_oth[:, 0] < d_min[:, 0]) * 100

shape_rows = []
for c in (TARGET_CATS if RUN_B_SHAPE else []):
    t0 = time.time()
    X_cat, X_other = X_tr[y_tr == c], X_tr[y_tr != c]
    nn_other = NearestNeighbors(n_neighbors=1).fit(X_other)
    reps = 1 if len(X_cat) <= N_CONTROL else REPEATS_SHAPE
    res = []
    for r in range(reps):
        pick = rng_shape.choice(len(X_cat), size=min(N_CONTROL, len(X_cat)), replace=False)
        X_min = X_cat[pick]
        row = example_types(X_min, X_other)
        row["dispersion ratio"] = dispersion_ratio(X_min, nn_other)
        row["synthetic intrusion (%)"] = synthetic_intrusion(X_min, nn_other, rng_shape)
        res.append(row)
    df = pd.DataFrame(res)
    out = {"Category": c, "n": min(N_CONTROL, len(X_cat)), "repeats": reps}
    for col in df.columns:
        out[col + " mean"] = df[col].mean()
        out[col + " sd"] = df[col].std(ddof=0)
    shape_rows.append(out)
    print(f"{c:13s} selesai ({time.time()-t0:.0f} s)")

shape_df = pd.DataFrame(shape_rows)
if not RUN_B_SHAPE:
    raise SystemExit("Bagian B dilewati (fungsi tetap terdefinisi).")
shape_df.to_csv(os.path.join(OUTDIR, "secB_distribution_shape_n29_v2.csv"), index=False)

def pm(df, col, nd=1):
    return df[col + " mean"].map(lambda v: f"{v:.{nd}f}") + " ± " + df[col + " sd"].map(lambda v: f"{v:.{nd}f}")
show = pd.DataFrame({"Category": shape_df["Category"],
                     "Safe (%)": pm(shape_df, "safe (%)"), "Borderline (%)": pm(shape_df, "borderline (%)"),
                     "Rare (%)": pm(shape_df, "rare (%)"), "Outlier (%)": pm(shape_df, "outlier (%)"),
                     "Dispersion ratio": pm(shape_df, "dispersion ratio", 2),
                     "Synthetic intrusion (%)": pm(shape_df, "synthetic intrusion (%)")})
print(show.to_string(index=False))

Botnet        selesai (36 s)
PortScan      selesai (25 s)
Web Attack    selesai (36 s)
Infiltration  selesai (2 s)
DoS           selesai (27 s)
DDoS          selesai (26 s)
    Category    Safe (%) Borderline (%)    Rare (%) Outlier (%) Dispersion ratio Synthetic intrusion (%)
      Botnet 33.8 ± 11.8     24.8 ± 9.7 15.2 ± 10.2  26.2 ± 8.6      1.84 ± 0.85              35.0 ± 6.9
    PortScan  41.0 ± 9.8     11.4 ± 8.2  15.2 ± 8.6  32.4 ± 7.1      1.23 ± 0.58              25.8 ± 5.9
  Web Attack  13.6 ± 8.7     57.1 ± 8.8  22.1 ± 9.0   7.2 ± 4.7      2.19 ± 2.25              11.6 ± 4.3
Infiltration  34.5 ± 0.0     20.7 ± 0.0  17.2 ± 0.0  27.6 ± 0.0      0.41 ± 0.00              14.8 ± 0.0
         DoS  54.3 ± 9.3     10.2 ± 7.8   9.8 ± 7.2  25.7 ± 5.8      0.76 ± 0.20              10.1 ± 3.6
        DDoS  67.4 ± 8.8     18.3 ± 7.1   5.5 ± 3.5   8.8 ± 3.5      0.45 ± 0.15               1.5 ± 0.8


## C. Eksperimen terkontrol v2: base vs SMOTE vs random oversampling (ROS)

Protokol sama dengan v1: stratified 5-fold CV; di tiap fold hanya kategori target yang dipangkas ke 29 sampel latih. SMOTE dan ROS menaikkan setiap kelas di bawah 24.000 ke 24.000, sehingga keduanya menghasilkan jumlah data latih yang identik. Bedanya hanya pada **cara** menambah sampel.

Untuk setiap subsampel 29 yang dipakai, bentuk distribusinya dihitung dan disimpan sehingga bisa dikorelasikan dengan manfaat SMOTE di fold yang sama. Hasil base/SMOTE dari v1 dipakai ulang karena subsampelnya identik.

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler

def rf(seed, **kw):
    return RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed, **kw)

def smote_strategy(y):
    vc = pd.Series(y).value_counts()
    return {c: TARGET for c, n in vc.items() if n < TARGET}

def counts_for(y_true, y_pred, c):
    return (int(((y_pred == c) & (y_true == c)).sum()),
            int(((y_pred == c) & (y_true != c)).sum()),
            int(((y_pred != c) & (y_true == c)).sum()))

CK5 = os.path.join(OUTDIR, "sec5_controlled_counts.csv")           # dari v1
CKC = os.path.join(OUTDIR, "secC_controlled_v2.csv")
CKS = os.path.join(OUTDIR, "secC_subsample_shape.csv")
if os.path.exists(CKC):
    doneC = pd.read_csv(CKC)
else:
    doneC = pd.DataFrame(columns=["seed","fold","category","method","n_train","tp","fp","fn","fn_dest"])
    if os.path.exists(CK5):                                        # impor hasil v1
        v1 = pd.read_csv(CK5)
        v1["method"] = v1["smote"].map({0: "base", 1: "smote"}); v1["fn_dest"] = ""
        doneC = v1[doneC.columns]
        doneC.to_csv(CKC, index=False)
doneS = pd.read_csv(CKS) if os.path.exists(CKS) else pd.DataFrame()
keysC = set(zip(doneC.seed, doneC.fold, doneC.category, doneC.method))
keysS = set(zip(doneS.seed, doneS.fold, doneS.category)) if len(doneS) else set()
METHODS_C = ["base", "smote", "ros"]
total = len(SEEDS) * N_FOLDS * len(TARGET_CATS) * len(METHODS_C)
print(f"Sudah selesai {len([k for k in keysC if k[0] in SEEDS and k[2] in TARGET_CATS])} dari {total} model")

t_start = time.time(); n_run = 0
for seed in (SEEDS if RUN_C_CONTROLLED else []):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        need_fold = any((seed, fold, c, m) not in keysC for c in TARGET_CATS for m in METHODS_C) or \
                    any((seed, fold, c) not in keysS for c in TARGET_CATS)
        if not need_fold:
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            rng_t = np.random.default_rng(seed * 100 + fold)       # identik dengan v1
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            if (seed, fold, c) not in keysS:
                X_min, X_oth = Xtr_f[idx_c], Xtr_f[ytr_f != c]
                nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
                srow = dict(seed=seed, fold=fold, category=c, **example_types(X_min, X_oth))
                srow["dispersion ratio"] = dispersion_ratio(X_min, nn_o)
                srow["synthetic intrusion (%)"] = synthetic_intrusion(X_min, nn_o, np.random.default_rng(seed + fold))
                doneS = pd.concat([doneS, pd.DataFrame([srow])], ignore_index=True)
                doneS.to_csv(CKS, index=False); keysS.add((seed, fold, c))
            for mth in METHODS_C:
                if (seed, fold, c, mth) in keysC:
                    continue
                t0 = time.time()
                if mth == "smote":
                    Xs, ys = SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5, random_state=seed).fit_resample(Xk, yk)
                elif mth == "ros":
                    Xs, ys = RandomOverSampler(sampling_strategy=smote_strategy(yk), random_state=seed).fit_resample(Xk, yk)
                else:
                    Xs, ys = Xk, yk
                pred = rf(seed).fit(Xs, ys).predict(Xte_f)
                tp, fp, fn = counts_for(yte_f, pred, c)
                dest = pd.Series(pred[(yte_f == c) & (pred != c)]).value_counts().to_dict()
                row = dict(seed=seed, fold=fold, category=c, method=mth, n_train=len(idx_c),
                           tp=tp, fp=fp, fn=fn, fn_dest=json.dumps(dest))
                doneC = pd.concat([doneC, pd.DataFrame([row])], ignore_index=True)
                doneC.to_csv(CKC, index=False); keysC.add((seed, fold, c, mth)); n_run += 1
                left = total - len([k for k in keysC if k[0] in SEEDS and k[2] in TARGET_CATS])
                avg = (time.time() - t_start) / n_run
                print(f"seed {seed} fold {fold} {c:12s} {mth:5s} {time.time()-t0:5.0f} s | sisa {left} ≈ {left*avg/60:.0f} menit")
print("Bagian C selesai.")

Sudah selesai 120 dari 270 model
seed 42 fold 0 Botnet       ros      27 s | sisa 149 ≈ 72 menit
seed 42 fold 0 PortScan     ros      28 s | sisa 148 ≈ 71 menit
seed 42 fold 0 Web Attack   ros      27 s | sisa 147 ≈ 71 menit
seed 42 fold 0 Infiltration ros      28 s | sisa 146 ≈ 70 menit
seed 42 fold 0 DoS          base     14 s | sisa 145 ≈ 63 menit
seed 42 fold 0 DoS          smote    47 s | sisa 144 ≈ 71 menit
seed 42 fold 0 DoS          ros      23 s | sisa 143 ≈ 68 menit
seed 42 fold 0 DDoS         base     13 s | sisa 142 ≈ 64 menit
seed 42 fold 0 DDoS         smote    47 s | sisa 141 ≈ 69 menit
seed 42 fold 0 DDoS         ros      23 s | sisa 140 ≈ 67 menit
seed 42 fold 1 Botnet       ros      26 s | sisa 139 ≈ 66 menit
seed 42 fold 1 PortScan     ros      27 s | sisa 138 ≈ 66 menit
seed 42 fold 1 Web Attack   ros      26 s | sisa 137 ≈ 65 menit
seed 42 fold 1 Infiltration ros      33 s | sisa 136 ≈ 66 menit
seed 42 fold 1 DoS          base     14 s | sisa 135 ≈ 63 menit
seed 42

## D. Tabel 4 lengkap pada Infiltration

Sama dengan bagian 6 di v1, sekarang dengan semua konfigurasi. Hasil baseline, SMOTE, dan Isolation Forest dari v1 dipakai ulang.

In [ ]:
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import RandomOverSampler, BorderlineSMOTE, ADASYN, SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier

UNDER_TO = 5000
INF = "Infiltration"

def fit_predict_proba(method, Xtr, ytr, Xte, seed):
    strat = smote_strategy(ytr)
    if method == "Baseline (no handling)":
        m = rf(seed).fit(Xtr, ytr)
    elif method == "Cost-sensitive learning":
        m = rf(seed, class_weight="balanced").fit(Xtr, ytr)
    elif method == "Random oversampling":
        m = rf(seed).fit(*RandomOverSampler(sampling_strategy=strat, random_state=seed).fit_resample(Xtr, ytr))
    elif method == "SMOTE":
        m = rf(seed).fit(*SMOTE(sampling_strategy=strat, k_neighbors=5, random_state=seed).fit_resample(Xtr, ytr))
    elif method == "Borderline-SMOTE":
        m = rf(seed).fit(*BorderlineSMOTE(sampling_strategy=strat, k_neighbors=5, random_state=seed).fit_resample(Xtr, ytr))
    elif method == "ADASYN":
        # ADASYN dijalankan per kelas; kelas yang tidak bisa diproses ADASYN (tidak punya
        # tetangga kelas lain) dibiarkan apa adanya dan dicatat.
        Xr, yr = Xtr, ytr
        for c_name, tgt in strat.items():
            try:
                Xr, yr = ADASYN(sampling_strategy={c_name: tgt}, n_neighbors=5,
                                random_state=seed).fit_resample(Xr, yr)
            except (RuntimeError, ValueError) as e:
                print(f"   ADASYN dilewati untuk kelas {c_name}: {str(e)[:60]}")
        m = rf(seed).fit(Xr, yr)
    elif method == "SMOTE-Tomek":
        st = SMOTETomek(smote=SMOTE(sampling_strategy=strat, k_neighbors=5, random_state=seed), random_state=seed, n_jobs=-1)
        m = rf(seed).fit(*st.fit_resample(Xtr, ytr))
    elif method == "Moderate undersampling":
        vc = pd.Series(ytr).value_counts()
        us = {c: UNDER_TO for c, n in vc.items() if n > UNDER_TO}
        m = rf(seed).fit(*RandomUnderSampler(sampling_strategy=us, random_state=seed).fit_resample(Xtr, ytr))
    elif method == "Balanced Random Forest":
        m = BalancedRandomForestClassifier(n_estimators=RF_TREES, sampling_strategy="all", replacement=True,
                                           bootstrap=False, n_jobs=-1, random_state=seed).fit(Xtr, ytr)
    else:
        raise ValueError(method)
    return m.classes_, m.predict_proba(Xte)

FAST = ["Baseline (no handling)", "SMOTE"]
FULL = ["Cost-sensitive learning", "Random oversampling", "ADASYN", "Baseline (no handling)", "SMOTE",
        "SMOTE-Tomek", "Borderline-SMOTE", "Moderate undersampling", "Balanced Random Forest"]
MODEL_METHODS = FULL if RUN_ALL_METHODS else FAST

CK6 = os.path.join(OUTDIR, "sec6_methods_counts.csv")
done6 = pd.read_csv(CK6) if os.path.exists(CK6) else pd.DataFrame(
    columns=["seed","fold","method","inf_tp","inf_fp","inf_fn","benign_tp","benign_fn"])
keys6 = set(zip(done6.seed, done6.fold, done6.method))

def record(seed, fold, method, yte, pred):
    global done6
    tp, fp, fn = counts_for(yte, pred, INF)
    btp = int(((pred == "BENIGN") & (yte == "BENIGN")).sum())
    bfn = int(((pred != "BENIGN") & (yte == "BENIGN")).sum())
    row = dict(seed=seed, fold=fold, method=method, inf_tp=tp, inf_fp=fp, inf_fn=fn, benign_tp=btp, benign_fn=bfn)
    done6 = pd.concat([done6, pd.DataFrame([row])], ignore_index=True)
    done6.to_csv(CK6, index=False); keys6.add((seed, fold, method))

for seed in (SEEDS if RUN_D_METHODS else []):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        all_need = [m for m in MODEL_METHODS + ["Isolation Forest"] + (["Threshold moving (x10)", "Threshold moving (x50)"] if RUN_ALL_METHODS else [])]
        if all((seed, fold, m) in keys6 for m in all_need):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for method in MODEL_METHODS:
            need = [method] + (["Threshold moving (x10)", "Threshold moving (x50)"] if method == "Baseline (no handling)" and RUN_ALL_METHODS else [])
            missing = [m for m in need if (seed, fold, m) not in keys6]
            if not missing:
                continue
            t0 = time.time()
            classes, proba = fit_predict_proba(method, Xtr_f, ytr_f, Xte_f, seed)
            if method in missing:
                record(seed, fold, method, yte_f, classes[proba.argmax(axis=1)])
            if method == "Baseline (no handling)" and RUN_ALL_METHODS:
                j = list(classes).index(INF)
                for mult in (10, 50):
                    if f"Threshold moving (x{mult})" in missing:
                        p2 = proba.copy(); p2[:, j] *= mult
                        record(seed, fold, f"Threshold moving (x{mult})", yte_f, classes[p2.argmax(axis=1)])
            print(f"seed {seed} fold {fold} {method:26s} {time.time()-t0:5.0f} s")
        if (seed, fold, "Isolation Forest") not in keys6:
            iso = IsolationForest(n_estimators=RF_TREES, contamination="auto", random_state=seed, n_jobs=-1)
            iso.fit(Xtr_f[ytr_f == "BENIGN"])
            sel = np.isin(yte_f, ["BENIGN", INF])
            flag = iso.predict(Xte_f[sel]) == -1
            pred = np.where(flag, INF, "BENIGN")
            record(seed, fold, "Isolation Forest", yte_f[sel], pred)
            print(f"seed {seed} fold {fold} Isolation Forest")
print("Bagian 6 selesai.")

seed 42 fold 0 Cost-sensitive learning       38 s
seed 42 fold 0 Random oversampling           27 s
seed 42 fold 0 ADASYN                        59 s
seed 42 fold 0 Baseline (no handling)        20 s
seed 42 fold 0 SMOTE-Tomek                  304 s
seed 42 fold 0 Borderline-SMOTE              57 s
seed 42 fold 0 Moderate undersampling         4 s
seed 42 fold 0 Balanced Random Forest         2 s
seed 42 fold 1 Cost-sensitive learning       21 s
seed 42 fold 1 Random oversampling           28 s
seed 42 fold 1 ADASYN                        54 s
seed 42 fold 1 Baseline (no handling)        21 s
seed 42 fold 1 SMOTE-Tomek                  282 s
seed 42 fold 1 Borderline-SMOTE              54 s
seed 42 fold 1 Moderate undersampling         4 s
seed 42 fold 1 Balanced Random Forest         2 s
seed 42 fold 2 Cost-sensitive learning       20 s
seed 42 fold 2 Random oversampling           26 s
seed 42 fold 2 ADASYN                        58 s
seed 42 fold 2 Baseline (no handling)        21 s


## E. CSE-CIC-IDS2018: kurva belajar Infiltration (Tabel 5, Fig. 3)

Hanya dua hari yang berisi Infiltration yang diunduh (Rabu 28-02-2018 dan Kamis 01-03-2018) dari bucket publik AWS milik CIC. Pipeline pembersihan sama dengan CIC-IDS2017. Label di dataset ini tertulis **"Infilteration"** (salah eja resmi), dan kodenya sudah menangani itu.

Protokol: dari kumpulan BENIGN dan Infiltration (jumlah per kelas disamakan), 20% disisihkan sebagai data uji tetap (1:1). Model dilatih dengan ukuran per kelas yang bertambah (1.000 → maksimum), diulang untuk setiap seed.

Jika unduhan otomatis gagal, unduh kedua file CSV secara manual dari https://www.unb.ca/cic/datasets/ids-2018.html, taruh di `Drive/juti_rerun/cse2018/`, lalu jalankan ulang sel ini.

In [ ]:
CKE = os.path.join(OUTDIR, "secE_cse2018_curve.csv")
CSE_DIR = os.path.join(OUTDIR, "cse2018"); os.makedirs(CSE_DIR, exist_ok=True)
SIZES = [1000, 5000, 10000, 24000, 40000, 60000, 80000]      # ukuran maksimum ditambahkan otomatis

def fetch_cse2018():
    have = sorted(glob.glob(os.path.join(CSE_DIR, "*.csv")))
    if len(have) >= 2 or TEST_MODE:
        return have
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="ca-central-1")
    prefix = "Processed Traffic Data for ML Algorithms/"
    keys = [o["Key"] for o in s3.list_objects_v2(Bucket="cse-cic-ids2018", Prefix=prefix).get("Contents", [])]
    want = [k for k in keys if ("28-02-2018" in k or "01-03-2018" in k) and k.endswith(".csv")]
    print("File yang diunduh:", want)
    for k in want:
        dst = os.path.join(CSE_DIR, os.path.basename(k))
        if not os.path.exists(dst):
            s3.download_file("cse-cic-ids2018", k, dst)
    return sorted(glob.glob(os.path.join(CSE_DIR, "*.csv")))

if RUN_E_CSE2018:
    files18 = fetch_cse2018()
    if not files18:
        raise RuntimeError("File CSE-CIC-IDS2018 tidak ditemukan. Unggah manual ke Drive/juti_rerun/cse2018/")
    fr = []
    for f in files18:
        d = pd.read_csv(f, low_memory=False)
        d.columns = [c.strip() for c in d.columns]
        fr.append(d)
    raw18 = pd.concat(fr, ignore_index=True); del fr
    raw18 = raw18[raw18["Label"].astype(str).str.strip() != "Label"]          # baris header berulang
    raw18 = raw18.drop_duplicates()
    lab18 = raw18["Label"].astype(str).str.strip()
    lab18 = lab18.replace({"Benign": "BENIGN", "Infilteration": "Infiltration", "Infiltration": "Infiltration"})
    X18 = raw18.drop(columns=["Label"]); del raw18
    for c_ in X18.select_dtypes(include="object").columns:
        X18[c_] = pd.to_numeric(X18[c_], errors="coerce")
    X18 = X18.loc[:, X18.notna().any()].replace([np.inf, -np.inf], np.nan)
    ok = X18.notna().all(axis=1) & lab18.isin(["BENIGN", "Infiltration"])
    X18, lab18 = X18[ok].reset_index(drop=True), lab18[ok].reset_index(drop=True)
    X18 = X18.drop(columns=[c_ for c_ in X18.columns if X18[c_].nunique() <= 1])
    print("CSE-CIC-IDS2018 setelah dibersihkan:", lab18.value_counts().to_dict(), "| fitur:", X18.shape[1])

    Xb = X18.values.astype(np.float64); yb = lab18.values
    n_pool = int(min((yb == "BENIGN").sum(), (yb == "Infiltration").sum()))
    doneE = pd.read_csv(CKE) if os.path.exists(CKE) else pd.DataFrame()
    for seed in SEEDS:
        rs = np.random.default_rng(seed)
        idx_b = rs.choice(np.where(yb == "BENIGN")[0], n_pool, replace=False)
        idx_i = rs.choice(np.where(yb == "Infiltration")[0], n_pool, replace=False)
        n_test = int(0.2 * n_pool)
        te_idx = np.r_[idx_b[:n_test], idx_i[:n_test]]
        pool_b, pool_i = idx_b[n_test:], idx_i[n_test:]
        max_train = len(pool_b)
        sizes = sorted(set([s for s in SIZES if s < max_train] + [max_train]))
        for n in sizes:
            if len(doneE) and ((doneE.seed == seed) & (doneE.n_per_class == n)).any():
                continue
            tr_idx = np.r_[pool_b[:n], pool_i[:n]]
            sc = StandardScaler().fit(Xb[tr_idx])
            t0 = time.time()
            pred = rf(seed).fit(sc.transform(Xb[tr_idx]), yb[tr_idx]).predict(sc.transform(Xb[te_idx]))
            yt = yb[te_idx]
            tp, fp, fn = counts_for(yt, pred, "Infiltration")
            row = dict(seed=seed, n_per_class=n, n_test_per_class=n_test, tp=tp, fp=fp, fn=fn,
                       benign_recall=100 * np.mean(pred[yt == "BENIGN"] == "BENIGN"),
                       accuracy=100 * np.mean(pred == yt), n_features=X18.shape[1],
                       n_infiltration_total=int((yb == "Infiltration").sum()))
            doneE = pd.concat([doneE, pd.DataFrame([row])], ignore_index=True); doneE.to_csv(CKE, index=False)
            print(f"seed {seed} n={n:>7,}  recall {100*tp/(tp+fn):.1f}  precision {100*tp/max(tp+fp,1):.1f}  ({time.time()-t0:.0f} s)")

File yang diunduh: ['Processed Traffic Data for ML Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv', 'Processed Traffic Data for ML Algorithms/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv']
CSE-CIC-IDS2018 setelah dibersihkan: {'BENIGN': 769702, 'Infiltration': 160604} | fitur: 70
seed 42 n=  1,000  recall 53.2  precision 55.0  (1 s)
seed 42 n=  5,000  recall 54.2  precision 56.1  (4 s)
seed 42 n= 10,000  recall 54.6  precision 57.0  (11 s)
seed 42 n= 24,000  recall 55.2  precision 57.1  (23 s)
seed 42 n= 40,000  recall 56.2  precision 57.6  (39 s)
seed 42 n= 60,000  recall 56.4  precision 57.4  (59 s)
seed 42 n= 80,000  recall 57.0  precision 57.5  (80 s)
seed 42 n=128,484  recall 57.7  precision 57.5  (133 s)
seed 43 n=  1,000  recall 53.3  precision 55.2  (2 s)
seed 43 n=  5,000  recall 55.5  precision 56.2  (6 s)
seed 43 n= 10,000  recall 55.7  precision 56.9  (10 s)
seed 43 n= 24,000  recall 56.3  precision 57.4  (24 s)
seed 43 n= 40,000  recall 56.9  precision 

## F. Ringkasan untuk naskah

Menyimpan ke Drive:
* `v2_table1.csv`, `v2_table6.csv`, `v2_table3_methods.csv`, `v2_table4.csv`, `v2_table5.csv`, `v2_shape.csv`, `v2_correlations.csv`
* `v2_fig_mechanism.png` dan `v2_fig_cse2018.png` (label bahasa Inggris, 300 dpi)

**Kirim seluruh file `v2_*.csv`, `preprocessing_summary.json`, dan kedua PNG.**

In [ ]:
def wilson(k, n, z=1.959964):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return 100*(c-h), 100*(c+h)

def f1_from(tp, fp, fn):
    return 0.0 if tp == 0 else 200 * tp / (2*tp + fp + fn)


from scipy.stats import spearmanr
import matplotlib; matplotlib.use("Agg") if TEST_MODE else None
import matplotlib.pyplot as plt
SAVE = lambda df, name: (df.to_csv(os.path.join(OUTDIR, name), index=True), print(f"\n=== {name} ===\n" + df.to_string()))

# ---- Tabel 1 & 6
if os.path.exists(CKA):
    a = pd.read_csv(CKA)
    t1 = a.set_index("algorithm")[["accuracy", "macro_f1", "train_time_s"]].sort_values("accuracy", ascending=False).round(2)
    SAVE(t1, "v2_table1.csv")
    t6 = a.set_index("algorithm")[["inf_correct", "inf_test", "inf_fp", "inf_fn_to_benign"]].copy()
    t6["recall (%)"] = (100 * t6.inf_correct / t6.inf_test).round(1)
    t6["Wilson 95% (%)"] = ["%.1f–%.1f" % wilson(k, n) for k, n in zip(t6.inf_correct, t6.inf_test)]
    SAVE(t6.sort_values("recall (%)", ascending=False), "v2_table6.csv")

# ---- Tabel 3 v2 (base / SMOTE / ROS) + FN destination
if os.path.exists(CKC):
    dc = pd.read_csv(CKC); dc = dc[dc.seed.isin(SEEDS) & dc.category.isin(TARGET_CATS)]
    ps = dc.groupby(["category", "method", "seed"])[["tp", "fp", "fn"]].sum().reset_index()
    ps["f1"] = [f1_from(*r) for r in ps[["tp", "fp", "fn"]].values]
    ps["recall"] = 100 * ps.tp / (ps.tp + ps.fn); ps["precision"] = 100 * ps.tp / (ps.tp + ps.fp).clip(lower=1)
    wide = ps.pivot_table(index=["category", "seed"], columns="method", values="f1")
    for m in ["smote", "ros"]:
        if m in wide: wide["d_" + m] = wide[m] - wide["base"]
    agg = wide.groupby("category").agg(["mean", "std"]).round(1)
    agg.columns = [f"{a_}_{b_}" for a_, b_ in agg.columns]
    rp = ps.groupby(["category", "method"])[["recall", "precision"]].mean().unstack().round(1)
    rp.columns = [f"{a_}_{b_}" for a_, b_ in rp.columns]
    t3 = agg.join(rp).reindex([c for c in TARGET_CATS if c in agg.index])
    SAVE(t3, "v2_table3_methods.csv")
    fd = dc[(dc.method == "base") & dc.fn_dest.fillna("").str.len().gt(2)]
    if len(fd):
        dest = {}
        for c, js in zip(fd.category, fd.fn_dest):
            for k, v in json.loads(js).items():
                dest.setdefault(c, {}); dest[c][k] = dest[c].get(k, 0) + v
        dd = pd.DataFrame(dest).T.fillna(0).astype(int)
        SAVE(dd, "v2_fn_destination_base.csv")

# ---- korelasi bentuk distribusi vs manfaat SMOTE
if os.path.exists(CKS) and os.path.exists(CKC):
    sh = pd.read_csv(CKS)
    ff = dc.copy(); ff["f1"] = [f1_from(*r) for r in ff[["tp", "fp", "fn"]].values]
    fw = ff.pivot_table(index=["seed", "fold", "category"], columns="method", values="f1").reset_index()
    fw["d_smote"] = fw["smote"] - fw["base"]
    if "ros" in fw: fw["d_ros"] = fw["ros"] - fw["base"]
    m = fw.merge(sh, on=["seed", "fold", "category"])
    metrics = ["safe (%)", "borderline (%)", "rare (%)", "outlier (%)", "dispersion ratio", "synthetic intrusion (%)"]
    rows = []
    cat_mean = m.groupby("category")[["d_smote", "base"] + metrics].mean()
    for met in metrics + ["base"]:
        r_between = spearmanr(cat_mean["d_smote"], cat_mean[met]).correlation
        within = []
        for c, g_ in m.groupby("category"):
            if g_[met].nunique() > 1 and g_["d_smote"].nunique() > 1:
                within.append(spearmanr(g_["d_smote"], g_[met]).correlation)
        rows.append({"metric": met if met != "base" else "baseline F1 (%)",
                     "rho between categories (n=%d)" % len(cat_mean): round(r_between, 2),
                     "median rho within category": round(float(np.nanmedian(within)), 2) if within else np.nan,
                     "pooled rho (all subsamples)": round(spearmanr(m["d_smote"], m[met]).correlation, 2)})
    SAVE(pd.DataFrame(rows).set_index("metric"), "v2_correlations.csv")
    shape_tab = m.groupby("category")[metrics].agg(["mean", "std"]).round(2)
    shape_tab.columns = [f"{a_} {b_}" for a_, b_ in shape_tab.columns]
    SAVE(shape_tab.reindex([c for c in TARGET_CATS if c in shape_tab.index]), "v2_shape.csv")

    # Gambar mekanisme
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), dpi=300)
    cats = [c for c in TARGET_CATS if c in agg.index]; xx = np.arange(len(cats)); w = 0.38
    ax[0].bar(xx - w/2, agg.loc[cats, "d_smote_mean"], w, yerr=agg.loc[cats, "d_smote_std"], capsize=3, label="SMOTE", color="#2e75a3")
    if "d_ros_mean" in agg:
        ax[0].bar(xx + w/2, agg.loc[cats, "d_ros_mean"], w, yerr=agg.loc[cats, "d_ros_std"], capsize=3, label="Random oversampling", color="#c0504d")
    ax[0].axhline(0, color="black", lw=0.8); ax[0].set_xticks(xx); ax[0].set_xticklabels(cats, rotation=20)
    ax[0].set_ylabel("F1 gain over no resampling (points)"); ax[0].set_title("(a) Gain at n = 29 training samples"); ax[0].legend()
    for c in cats:
        g_ = m[m.category == c]; ax[1].scatter(g_["synthetic intrusion (%)"], g_["d_smote"], s=18, label=c, alpha=0.8)
    ax[1].axhline(0, color="black", lw=0.8)
    ax[1].set_xlabel("Synthetic intrusion of the 29-sample subset (%)"); ax[1].set_ylabel("SMOTE gain in F1 (points)")
    ax[1].set_title("(b) Per-subsample geometry vs. SMOTE gain"); ax[1].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, "v2_fig_mechanism.png"), facecolor="white"); plt.close(fig)

# ---- Tabel 4
CK6 = os.path.join(OUTDIR, "sec6_methods_counts.csv")
if os.path.exists(CK6):
    d6 = pd.read_csv(CK6); d6 = d6[d6.seed.isin(SEEDS)]
    ps6 = d6.groupby(["method", "seed"])[["inf_tp", "inf_fp", "inf_fn", "benign_tp", "benign_fn"]].sum().reset_index()
    ps6["recall"] = 100 * ps6.inf_tp / (ps6.inf_tp + ps6.inf_fn)
    ps6["precision"] = [100 * t / (t + f) if (t + f) else 0.0 for t, f in zip(ps6.inf_tp, ps6.inf_fp)]
    ps6["f1"] = [f1_from(t, f, n) for t, f, n in zip(ps6.inf_tp, ps6.inf_fp, ps6.inf_fn)]
    ps6["benign_recall"] = 100 * ps6.benign_tp / (ps6.benign_tp + ps6.benign_fn)
    g6 = ps6.groupby("method")
    t4 = pd.DataFrame({"Recall (%)": g6.recall.mean(), "Precision (%)": g6.precision.mean(), "F1 (%)": g6.f1.mean(),
                       "BENIGN recall (%)": g6.benign_recall.mean(), "Detected (mean)": g6.inf_tp.mean(),
                       "Detected SD": g6.inf_tp.std(ddof=0), "False alarms (mean)": g6.inf_fp.mean(),
                       "Seeds": g6.seed.nunique()}).sort_values("F1 (%)", ascending=False)
    n_inf = int((y_all == "Infiltration").sum())
    t4["Wilson 95% (%)"] = ["%.1f–%.1f" % wilson(int(round(k)), n_inf) for k in t4["Detected (mean)"]]
    SAVE(t4.round(1), "v2_table4.csv")

# ---- Tabel 5
if os.path.exists(CKE):
    e = pd.read_csv(CKE); e = e[e.seed.isin(SEEDS)]
    e["recall"] = 100 * e.tp / (e.tp + e.fn); e["precision"] = 100 * e.tp / (e.tp + e.fp).clip(lower=1)
    e["f1"] = [f1_from(*r) for r in e[["tp", "fp", "fn"]].values]
    t5 = e.groupby("n_per_class")[["recall", "precision", "f1", "benign_recall", "accuracy"]].agg(["mean", "std"]).round(1)
    t5.columns = [f"{a_} {b_}" for a_, b_ in t5.columns]
    SAVE(t5, "v2_table5.csv")
    print("Infiltration total (setelah dedup):", int(e.n_infiltration_total.iloc[0]), "| fitur:", int(e.n_features.iloc[0]))
    fig, ax = plt.subplots(figsize=(7.5, 4), dpi=300)
    for col, lab, st in [("recall", "Infiltration recall", "-o"), ("precision", "Infiltration precision", "-.s"),
                         ("f1", "Infiltration F1", "--^"), ("benign_recall", "BENIGN recall", ":d")]:
        mu = e.groupby("n_per_class")[col].mean(); sd = e.groupby("n_per_class")[col].std().fillna(0)
        ax.errorbar(mu.index, mu.values, yerr=sd.values, fmt=st, capsize=2, label=lab, ms=4)
    ax.set_xscale("log"); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
    ax.set_xlabel("BENIGN and Infiltration training samples per class (1:1 ratio, log scale)"); ax.set_ylabel("Percentage (%)")
    ax.legend(loc="lower right", fontsize=8); fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "v2_fig_cse2018.png"), facecolor="white"); plt.close(fig)
print("\nSelesai. Kirim file v2_*.csv, preprocessing_summary.json, dan kedua PNG dari folder:", OUTDIR)


=== v2_table1.csv ===
                     accuracy  macro_f1  train_time_s
algorithm                                            
XGBoost                 99.90     98.65         34.57
Gradient Boosting       99.81     97.61        721.24
Random Forest           99.81     97.17         20.38
Decision Tree           99.76     95.83          2.56
Extra Trees             99.76     96.94          6.59
K-Nearest Neighbors     99.22     92.56          0.01
Linear SVC              96.64     79.24        555.11
Naive Bayes             90.34     71.81          0.12

=== v2_table6.csv ===
                     inf_correct  inf_test  inf_fp  inf_fn_to_benign  recall (%) Wilson 95% (%)
algorithm                                                                                      
XGBoost                        6         7       0                 1        85.7      48.7–97.4
Gradient Boosting              6         7       1                 1        85.7      48.7–97.4
Decision Tree                 